<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB_WEEK3_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주제** 영화 리뷰 감성분석 AI 해커톤

https://dacon.io/competitions/official/235864/codeshare/4265?page=1&dtype=recent

**코드 흐름**

텍스트 토큰화, 노이즈 제거
- 리뷰 내용을 id, document label preprocessed로 나열해 한눈에 볼 수 있도록 만듦.

**TfidfVectorizer와 모델을 연결**한 파이프라인을 반환하는 함수 제작.

모델을 입력받아 **KFold 예측 후 accuracy score를 반환**하는 함수 제작.
- naive_bayes, SGD, rfc, SVC, ada, lgbm, lgbm2, xgb, knc1, knc2의 모델을 입력해 accuracy score를 반환받음.

StackingClassifier의 ensemble을 이용해 스태킹 모델 제작.

**새롭게 알게 된 점/어려운 점/배울 점**

TfidfVectorizer와 모델을 연결하는 파이프라인을 만들어 이 모델들의 accuracy score를 통째로 반환받는 함수를 만드는 방법도 있다는 것을 알았다. 파이프라인을 요긴하게 쓰는 방법을 더 알아두는 것도 좋을 것 같다.

In [ ]:
# 텍스트 전처리

def preprocess(text):
  text = text.lower()
  text = re.sub("[^A-Za-z가-힣 ]","", text)  # re.sub(r"[^A-Za-zㄱ-ㅎㅏ-ㅣ가-힣 ]","", text)
  return text

train_df['preprocessed'] = train_df.document.apply(lambda x : preprocess(x))
test_df['preprocessed'] = test_df.document.apply(lambda x : preprocess(x))

train_df.head()

In [ ]:
def get_pipe(model, model_name: str) -> Pipeline:
    "TfidfVectorizer와 모델을 연결한 파이프라인을 반환하는 함수"
    tfidf = TfidfVectorizer(analyzer="char", ngram_range=(1, 3))
    pipe = Pipeline([
        ("tfidf", tfidf),
        (model_name, model)
    ])
    return pipe

In [ ]:
def return_kfold_accuarcy(model, k: int = 5) -> float:
    "모델을 입력받아 KFold 예측 후 accuracy score를 반환하는 함수"
    kfold = StratifiedKFold(k, shuffle=True, random_state=42)
    result = []
    for train_idx, test_idx in kfold.split(train_df["document"], train_df["label"]):
        train, val = train_df.iloc[train_idx], train_df.iloc[test_idx]
        model.fit(train["document"], train["label"])
        pred = model.predict(val["document"])
        acc = accuracy_score(val["label"], pred)
        result.append(acc)

    return np.mean(result)

In [ ]:
models = [
    ("naive_bayes", BernoulliNB()),
    ("SGD", SGDClassifier(random_state=42, n_jobs=-1)),
    ("rfc", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ("SVC", SVC(random_state=42)),
    ("ada", AdaBoostClassifier(random_state=42)),
    ("lgbm", LGBMClassifier(random_state=42)),
    ("lgbm2", LGBMClassifier(n_estimators=80, random_state=42)),
    ("xgb", XGBClassifier(random_state=42)),
    ("knc1", KNeighborsClassifier()),
    ("knc2", KNeighborsClassifier(n_neighbors=4))
]

model_pipes = [(name, get_pipe(model, name)) for name, model in models]